In [0]:
from pyspark.sql.functions import lit

# Read both parquet files that are already in the Volume
print("Reading parquet files from Volume...")

df_en = spark.read.parquet("/Volumes/arthasetu/bronze/uploads/English_test-00000-of-00001.parquet")
df_en = df_en.withColumn("lang", lit("en"))
print(f"  English: {df_en.count()} rows")

df_hi = spark.read.parquet("/Volumes/arthasetu/bronze/uploads/Hindi_test-00000-of-00001.parquet")
df_hi = df_hi.withColumn("lang", lit("hi"))
print(f"  Hindi: {df_hi.count()} rows")

# Combine them
df_combined = df_en.union(df_hi)
print(f"  Combined: {df_combined.count()} rows")

# Write to bronze table
print("\nWriting to arthasetu.bronze.bhashbench_finance_raw...")
df_combined.write.format("delta").mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable("arthasetu.bronze.bhashbench_finance_raw")

# Verify
cnt = spark.sql("SELECT COUNT(*) as c FROM arthasetu.bronze.bhashbench_finance_raw").collect()[0]['c']
print(f"✅ Created arthasetu.bronze.bhashbench_finance_raw → {cnt:,} rows")

# Show sample
print("\nSample data:")
spark.sql("SELECT id, question, topic, lang FROM arthasetu.bronze.bhashbench_finance_raw LIMIT 3").show(truncate=60)